# Juvanie medium — cloud тренинг (QLoRA, безплатен Colab T4)

**Какво прави този notebook:**
1. Инсталира нужните библиотеки
2. Зарежда `huihui-ai/Qwen2.5-7B-Instruct-abliterated-v2` в 4-bit
3. Тренира LoRA върху 93-та примера на Juvanie (приложението, MCP сървърите, устройствата, MAC адресите, Term49/Termux/Kali, BG/EN)
4. Конвертира LoRA-та в **GGUF** (готова за `llama-server --lora` на :8082)
5. Сваля файла `juvanie-lora-v1.gguf` на компютъра ти

**Стъпки:** Runtime → Run all (или Ctrl+F9). Изчакай ~30–40 мин. Накрая файлът се сваля сам.
Ако Colab пита за GPU: Runtime → Change runtime type → **T4 GPU**.


In [ ]:
!pip -q install -U transformers peft trl bitsandbytes datasets accelerate sentencepiece
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "НЯМА GPU — смени runtime на T4!")


In [ ]:
import json, subprocess
URL = "https://files.catbox.moe/c86218.jsonl"
subprocess.run(["curl","-sL","--retry","3","-A","Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120 Safari/537.36",URL,"-o","/content/dataset.jsonl"], check=True)
rows = [json.loads(l) for l in open("/content/dataset.jsonl", encoding="utf-8") if l.strip()]
print(f"Датасет: {len(rows)} примера")
assert len(rows) == 93


In [ ]:
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import torch, json

MODEL = "huihui-ai/Qwen2.5-7B-Instruct-abliterated-v2"
rows = [json.loads(l) for l in open("/content/dataset.jsonl", encoding="utf-8")]
ds = Dataset.from_list(rows)

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

def fmt(ex):
    return {"text": tok.apply_chat_template(ex["messages"], tokenize=False)}

cfg = SFTConfig(
    output_dir="/content/lora-out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=1,
    save_strategy="no",
    max_length=1024,
    bf16=True,
    report_to=[],
    dataset_text_field="text",
)

trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds.map(fmt),
                     peft_config=lora, processing_class=tok)
trainer.train()
trainer.save_model("/content/lora-out")
tok.save_pretrained("/content/lora-out")
print("LoRA готова: /content/lora-out")


In [ ]:
# Бърз тест още в Colab — виждаш разликата веднага
from peft import PeftModel
import torch
model.config.use_cache = True
ft = PeftModel.from_pretrained(model, "/content/lora-out")
ft.eval()

qs = [
    "колко ssh keys използваме за приложението",
    "колко виртуалки са горе",
    "какъв е MAC адресът на Z30-1",
]
for q in qs:
    msgs = [{"role":"system","content":rows[0]["messages"][0]["content"]},
            {"role":"user","content":q}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = ft.generate(ids, max_new_tokens=200, do_sample=False)
    print("="*60)
    print("ВЪПРОС:", q)
    print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))


In [ ]:
# Конвертиране на LoRA -> GGUF (за llama-server --lora на :8082)
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements.txt
!python /content/llama.cpp/convert_lora_to_gguf.py /content/lora-out --outfile /content/juvanie-lora-v1.gguf --outtype f16
import os
print("GGUF размер:", round(os.path.getsize("/content/juvanie-lora-v1.gguf")/1e6, 1), "MB")
from google.colab import files
files.download("/content/juvanie-lora-v1.gguf")


## След свалянето
Дай файла `juvanie-lora-v1.gguf` на Kimi (плъзни го в чата или го сложи в workspace) — той ще го качи на .195, ще го закачи с `--lora` на medium (:8082) и ще пусне eval преди/после.
